### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\dky42\AppData\Local\Temp\ipykernel_31324\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Agentic AI\Agentic_AI_Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory

from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: pdf1.pdf
  ✓ Loaded 23 pages

Total documents loaded: 23


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-22T05:28:04+00:00', 'moddate': '2026-04-22T05:28:04+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Latest Trends in Biomedical \nInstrumentation'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-22T05:28:04+00:00', 'moddate': '2026-04-22T05:28:04+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Introduction\n• Biomedical Instrumentation = medical devices +\nelectronics + sensors\n• Used for diagnosis, monitoring, therapy\n• Rapid evolution due to:\n– AI & Data Science\n– IoT\n– Miniaturization\n• Focus: Smart, real-time, patient-centric healthcare'),
 Document(metadata={'producer': 'www

In [5]:
### Text splitting get into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 23 documents into 18 chunks

Example chunk:
Content: Latest Trends in Biomedical 
Instrumentation...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-22T05:28:04+00:00', 'moddate': '2026-04-22T05:28:04+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-22T05:28:04+00:00', 'moddate': '2026-04-22T05:28:04+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Latest Trends in Biomedical \nInstrumentation'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-22T05:28:04+00:00', 'moddate': '2026-04-22T05:28:04+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Introduction\n• Biomedical Instrumentation = medical devices +\nelectronics + sensors\n• Used for diagnosis, monitoring, therapy\n• Rapid evolution due to:\n– AI & Data Science\n– IoT\n– Miniaturization\n• Focus: Smart, real-time, patient-centric healthcare'),
 Document(metadata={'producer': 'www

### Embedding and Vector Store DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2227.87it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\dky42\AppData\Local\Temp\ipykernel_31324\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 18


In [10]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-22T05:28:04+00:00', 'moddate': '2026-04-22T05:28:04+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Latest Trends in Biomedical \nInstrumentation'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2026-04-22T05:28:04+00:00', 'moddate': '2026-04-22T05:28:04+00:00', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Introduction\n• Biomedical Instrumentation = medical devices +\nelectronics + sensors\n• Used for diagnosis, monitoring, therapy\n• Rapid evolution due to:\n– AI & Data Science\n– IoT\n– Miniaturization\n• Focus: Smart, real-time, patient-centric healthcare'),
 Document(metadata={'producer': 'www

In [11]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 18 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]

Generated embeddings with shape: (18, 384)
Adding 18 documents to vector store...
Successfully added 18 documents to vector store
Total documents in collection: 36


### Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("What are the latest trends in biomedical engineering?")

Retrieving documents for query: 'What are the latest trends in biomedical engineering?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.03it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_309456ba_0',
  'content': 'Latest Trends in Biomedical \nInstrumentation',
  'metadata': {'file_type': 'pdf',
   'creationdate': '2026-04-22T05:28:04+00:00',
   'source': '..\\data\\pdf\\pdf1.pdf',
   'producer': 'www.ilovepdf.com',
   'page_label': '1',
   'doc_index': 0,
   'total_pages': 23,
   'content_length': 44,
   'moddate': '2026-04-22T05:28:04+00:00',
   'page': 0,
   'source_file': 'pdf1.pdf',
   'creator': 'Microsoft® PowerPoint® 2016'},
  'similarity_score': 0.21853220462799072,
  'distance': 0.7814677953720093,
  'rank': 1},
 {'id': 'doc_4f34e191_0',
  'content': 'Latest Trends in Biomedical \nInstrumentation',
  'metadata': {'doc_index': 0,
   'total_pages': 23,
   'producer': 'www.ilovepdf.com',
   'source_file': 'pdf1.pdf',
   'creationdate': '2026-04-22T05:28:04+00:00',
   'page_label': '1',
   'creator': 'Microsoft® PowerPoint® 2016',
   'moddate': '2026-04-22T05:28:04+00:00',
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\pdf1.pdf',
   'page': 0,
 

### Integration of Vector DB Context pipeline with LLM Output

In [15]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    api_key=groq_api_key,
    model="qwen/qwen3-32b"
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):

    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using Groq LLM

    prompt = """Use the following context to answer the question concisely.

    Context:
    {context}

    Question: {query}

    Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [16]:
answer = rag_simple("list down the latest trends in biomedical engineering" , rag_retriever , llm)
print(answer)

Retrieving documents for query: 'list down the latest trends in biomedical engineering'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


<think>
Okay, the user wants a list of the latest trends in biomedical engineering based on the provided context. Let me start by reading through the context again to make sure I don't miss anything.

The introduction mentions that biomedical instrumentation combines medical devices, electronics, and sensors, used for diagnosis, monitoring, and therapy. The rapid evolution is due to AI and data science, IoT, and miniaturization, with a focus on smart, real-time, patient-centric healthcare.

So the key areas here are AI/Data Science, IoT, and miniaturization. Let me break down each of these into trends. 

First, AI and Data Science. The context says they're driving rapid evolution. So trends here might include AI-driven diagnostics, predictive analytics for patient outcomes, and personalized treatment plans using machine learning. Also, maybe real-time data processing for immediate feedback in monitoring devices.

Next, IoT. Integration with IoT would mean wearable devices that connect 

### Enhanced RAG Pipeline Features

In [18]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Latest trends in biomedical engineering", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Latest trends in biomedical engineering'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: <think>
Okay, the user is asking about the latest trends in biomedical engineering, and they provided a context that says "Latest Trends in Biomedical Instrumentation" twice. Hmm, maybe they just want me to focus on instrumentation since that's what's mentioned. Let me think about what's currently trending in that area.

First, I remember that wearable devices are a big thing. People are always using smartwatches and fitness trackers, so maybe that's part of it. Then there's AI in medical diagnostics. AI helps analyze data from instruments more accurately and quickly. Also, non-invasive technologies are important because they reduce patient discomfort. Things like optical sensors for blood sugar monitoring without needles.

Another area is miniaturization of devices. Smaller devices mean better integration with the body, like microchips or nanobots for targeted drug delivery. 3D printing is also a trend; it allows for custom prosthetics and implants tailored to individual patie

In [19]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Latest trends in biomedical engineering", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Latest trends in biomedical engineering'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.64it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Latest Trends in Biomedical 
Instrumentation

Latest Trends in Biomedical 
Instrumentation

Question: Latest trends in biomedical engineering

Answer:



Final Answer: <think>
Okay, I need to answer the question about the latest trends in biomedical engineering based on the given context. The context mentions "Latest Trends in Biomedical Instrumentation," which is a part of biomedical engineering. Let me think about what's currently trending in biomedical instrumentation.

First, I recall that biomedical instrumentation involves devices used for diagnosis, monitoring, and treatment. Recent trends might include wearable devices for continuous health monitoring. For example, smartwatches that track heart rate or blood oxygen levels. Then there's the development of miniaturized and implantable devices, like microsensors that can be placed inside the body. Also, integration with AI for data analysis is a big trend. AI helps in interpreting the data from these devices more accurately.

Another area could be non-invasive diagnostic tools. For instance, using optical sensors to detect blood sugar levels without a finger prick. 3D printing is 